# PRAG - Polish RAG #

In [3]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)

In [4]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [5]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
)

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
# import json
# info = milvus_client.describe_collection(COLLECTION_NAME)
# print(json.dumps(info, indent=2, default=str))

print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470467611085771050, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 470467784523120659, 'update_timestamp': 470467784523120659}


In [6]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./lab/milvus_db/data"

In [7]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [8]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

In [ ]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

In [ ]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [ ]:
# search
def search(model, query, client=milvus_client):
    ret = []
    for q in query:
        embedded_query = model.encode(q).tolist()
        ret.append(client.search(
            collection_name="rag_texts_and_embeddings",
            data=[embedded_query],
            limit=1,
            search_params={"metric_type": "L2"},
            output_fields=["text"]
        ))
    return ret


MAX_PREVIEW_CHARS = 60

queries = ["Czym jest sztuczna inteligencja", "Reinforcement Learning", "Jak smakuje ser ?"]
results = search(model, query=queries)

for query_idx, (q, result) in enumerate(zip(queries, results), start=1):
    print(f"=== Zapytanie {query_idx} ===")
    print(f"Treść zapytania: '{q}'")
    print()
    for rank, hit in enumerate(result[0], start=1):
        print(f"--- trafienie {rank} ---")
        print(f"id:       {hit['id']}")
        print(f"distance: {hit['distance']:.4f}")  # L2: mniejsze = bliżej
        print()
        text = hit["entity"]["text"]
        if len(text) > MAX_PREVIEW_CHARS:
            text = text[:MAX_PREVIEW_CHARS].rstrip() + " …"
        print(f"Odpowiedź (fragment z bazy):")
        print(text)
    print("-" * 60)
    print()

=== Zapytanie 1 ===
Treść zapytania: 'Czym jest sztuczna inteligencja'

--- trafienie 1 ---
id:       470467611085775431
distance: 29.1252

Odpowiedź (fragment z bazy):
Historia powstania
sztucznej inteligencji
7
W języku potoczn …
------------------------------------------------------------

=== Zapytanie 2 ===
Treść zapytania: 'Reinforcement Learning'

--- trafienie 1 ---
id:       470467611085775439
distance: 57.6567

Odpowiedź (fragment z bazy):
Sztuczna inteligencja
generatywna
1 5
W ekonomii powszechnie …
------------------------------------------------------------

=== Zapytanie 3 ===
Treść zapytania: 'Jak smakuje ser ?'

--- trafienie 1 ---
id:       470467611085775431
distance: 64.8199

Odpowiedź (fragment z bazy):
Historia powstania
sztucznej inteligencji
7
W języku potoczn …
------------------------------------------------------------



In [ ]:
import os
from pathlib import Path

key_path = Path("lab/milvus_db/google_api_key___wojciech")
# key_path = Path("lab/milvus_db/google_api_key___work")
os.environ["GEMINI_API_KEY"] = key_path.read_text().strip()  # równoważne  export VAR=secret

In [ ]:
import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

# MODEL = "gemini-2.5-flash"    obsolete
MODEL = "gemini-3.8-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

generate_response("jak się masz ?")   # problemy po stronie Serwerów - high demand

Error generating response: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
